In [1]:
import numpy as np

In [22]:
class Tensor:
    def __init__(self, data, parents=(), creator=None, req_grad=True):
        self.data = np.asarray(data, dtype=np.float32)
        self.grad = np.zeros_like(self.data)
        self.parents = parents
        self.creator = creator
        self.requires_grad = req_grad
    def __repr__(self):
        op_name = "empty" if self.creator is None else f"{type(self.creator).__name__}"
        return f"Data={self.data}, grad={self.grad}, op={op_name}, requires_grad={self.requires_grad}"
    def backward(self):
        topo_arr = []
        self.topo_sort(set(),set(), topo_arr)
        topo_arr.reverse()
        for i in range(len(topo_arr)):
            if topo_arr[i].creator is None:
                continue
            else:
                topo_arr[i].grad = np.zeros_like(topo_arr[i].data)
        self.grad = np.ones_like(self.data)
        for i in range(len(topo_arr)):
            if topo_arr[i].creator is None:
                continue
            parent_gradients = topo_arr[i].creator.backward(topo_arr[i].grad)
            for n,s in zip(topo_arr[i].parents, parent_gradients):
                n.grad += s
    def zero_grad(self):
        self.grad = np.zeros_like(self.data)

    def topo_sort(self, visiting, visited, result):
        visiting.add(self)
        for i in range(len(self.parents)):
            if self.parents[i] not in visited:
                self.parents[i].topo_sort(visiting, visited, result)
        visiting.remove(self)
        visited.add(self)
        result.append(self)
        return


        



In [23]:
import numpy as np


class operations:

    def forward(self, *args, **kwargs):
        raise NotImplementedError

    def backward(self, *args, **kwargs):
        raise NotImplementedError

    @classmethod
    def run(cls, *args, **kwargs):
        obj = cls()
        inputs = tuple(a.data for a in args)
        result = obj.forward(*inputs, **kwargs)
        obj.output = result
        return Tensor(result, parents=args, creator=obj)

class Mul(operations):
    def forward(self, x, y):
        self.x,self.y = x,y
        return self.x*self.y
    def backward(self, gradient):
        return tuple([self.y*gradient, self.x*gradient])

In [28]:
a = Tensor([2., 3.]); s = Tensor(10.)     # s is a scalar, shape ()
c = Mul.run(a, s)
c.backward()
print(s.grad)   

ValueError: non-broadcastable output operand with shape () doesn't match the broadcast shape (2,)

In [29]:
# Test F — multiply by a plain number
a = Tensor([2., 3.])
c = Mul.run(a, 2)
c.backward()
print(a.grad)    

AttributeError: 'int' object has no attribute 'data'